# Conflict Resolution for Co-Occurring Modifiers

Clinical text routinely triggers multiple modifiers on a single entity. cwyde's `conflict_resolver` 
component applies a principled precedence hierarchy rather than an ad-hoc list. Every decision 
is recorded in `cwyde_resolution_trace` for explainability.

In [1]:
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

def show_trace(ent):
    print(f"  Entity   : {ent.text!r}")
    print(f"  Category : {ent._.cwyde_assertion_category.value}")
    trace = ent._.cwyde_resolution_trace or []
    if trace:
        print("  Trace:")
        for step in trace:
            print(f"    {step}")
    else:
        print("  Trace: (empty — no conflict)")

## Case A: FAMILY + HISTORICAL → FAMILY wins

A family history sentence fires both the FAMILY modifier (different experiencer) and the HISTORICAL 
modifier (past tense). These operate on different axes — FAMILY is an *experiencer* override and 
takes unconditional precedence over temporal categories.

In [2]:
doc = nlp("Father had PE.")
print("Case A: FAMILY + HISTORICAL")
print("-" * 50)
for ent in doc.ents:
    show_trace(ent)

2026-04-28 22:32:36.389 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'Father' marked as sentence start (span begin)


2026-04-28 22:32:36.389 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(Father, True), (had, False), (PE, False), (., False)]


Case A: FAMILY + HISTORICAL
--------------------------------------------------
  Entity   : 'PE'
  Category : FAMILY
  Trace:
    {'step': 'category_mapper', 'medspacy': 'FAMILY', 'cwyde': <AssertionCategory.FAMILY: 'FAMILY'>}


## Case B: HYPOTHETICAL + DEFINITE_EXISTENCE → HYPOTHETICAL wins

Conditional sentences assert existence in a hypothetical world, not in the patient's actual state. 
HYPOTHETICAL overrides the existence-axis category because the frame of reference is not the 
clinical present.

In [3]:
doc = nlp("If PE is present, anticoagulate.")
print("Case B: HYPOTHETICAL + DEFINITE_EXISTENCE")
print("-" * 50)
for ent in doc.ents:
    show_trace(ent)

2026-04-28 22:32:36.458 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'If' marked as sentence start (span begin)


2026-04-28 22:32:36.459 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(If, True), (PE, False), (is, False), (present, False), (,, False), (anticoagulate, False), (., False)]


Case B: HYPOTHETICAL + DEFINITE_EXISTENCE
--------------------------------------------------
  Entity   : 'PE'
  Category : HYPOTHETICAL
  Trace:
    {'step': 'category_mapper', 'medspacy': 'HYPOTHETICAL', 'cwyde': <AssertionCategory.HYPOTHETICAL: 'HYPOTHETICAL'>}
    {'step': 'category_mapper', 'medspacy': 'DEFINITE_EXISTENCE', 'cwyde': <AssertionCategory.DEFINITE_EXISTENCE: 'DEFINITE_EXISTENCE'>}
    {'step': 'conflict_resolver', 'input': ['HYPOTHETICAL', 'DEFINITE_EXISTENCE'], 'result': 'HYPOTHETICAL'}


## Case C: INDICATION section + negation → INDICATION wins

When a document section is classified as INDICATION (e.g., a report's INDICATION header), 
sentence-level modifiers within it are overridden. The section propagator applies 
`override_existing=True` for INDICATION sections. "No PE" in an INDICATION section still 
means the study was ordered to evaluate PE — not that PE was ruled out.

In [4]:
# A minimal two-sentence document: INDICATION section header followed by negated sentence
report = "INDICATION: No PE. Patient with pleuritic chest pain."
doc = nlp(report)
print("Case C: INDICATION section header overrides sentence-level negation")
print("-" * 60)
for ent in doc.ents:
    if ent.text.upper() in ("PE", "PULMONARY EMBOLISM"):
        show_trace(ent)
        print(f"  Section inherited: {ent._.cwyde_section_inherited}")

2026-04-28 22:32:36.521 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'INDICATION' marked as sentence start (span begin)


2026-04-28 22:32:36.521 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 5 'Patient' marked as sentence start (span end next token)


2026-04-28 22:32:36.522 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 5 'Patient' marked as sentence start (span begin)


2026-04-28 22:32:36.522 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] Token/tag mapping: [(INDICATION, True), (:, False), (No, False), (PE, False), (., False), (Patient, True), (with, False), (pleuritic, False), (chest, False), (pain, False), (., False)]


Case C: INDICATION section header overrides sentence-level negation
------------------------------------------------------------
  Entity   : 'PE'
  Category : INDICATION
  Trace:
    {'step': 'category_mapper', 'medspacy': 'AMBIVALENT_EXISTENCE', 'cwyde': <AssertionCategory.AMBIVALENT_EXISTENCE: 'AMBIVALENT_EXISTENCE'>}
    {'step': 'category_mapper', 'medspacy': 'DEFINITE_NEGATED_EXISTENCE', 'cwyde': <AssertionCategory.DEFINITE_NEGATED_EXISTENCE: 'DEFINITE_NEGATED_EXISTENCE'>}
    {'step': 'conflict_resolver', 'input': ['AMBIVALENT_EXISTENCE', 'DEFINITE_NEGATED_EXISTENCE'], 'result': 'DEFINITE_NEGATED_EXISTENCE'}
    {'step': 'section_propagator', 'section': 'labs_and_studies', 'section_assertion': 'INDICATION', 'previous': 'DEFINITE_NEGATED_EXISTENCE', 'result': 'INDICATION', 'override_existing': True}
  Section inherited: True


## Case D: UNRESOLVED — gamen-validate dependency

Some conflicts cannot be resolved by the YAML precedence table. In v0.1 the fallback strategy 
returns `UNRESOLVED` for these cases; formal resolution requires gamen-validate 
(see notebook 05). UNRESOLVED is an explicit non-answer — better than a silent wrong answer.

In [5]:
# This sentence may produce UNRESOLVED depending on which modifiers co-fire.
# The exact result depends on the loaded interaction_rules.yaml.
test_sent = "There were no echocardiographic signs of right heart strain."
doc = nlp(test_sent)
print("Case D: checking for UNRESOLVED")
print("-" * 50)
if doc.ents:
    for ent in doc.ents:
        print(f"  Entity   : {ent.text!r}")
        print(f"  Category : {ent._.cwyde_assertion_category.value}")
        if ent._.cwyde_assertion_category.value == "UNRESOLVED":
            print("  → UNRESOLVED: conflict not resolvable without gamen-validate (see notebook 05)")
else:
    print("  (no entities matched — add a TargetRule for 'right heart strain' to exercise this case)")

2026-04-28 22:32:36.585 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 0 'There' marked as sentence start (span begin)


2026-04-28 22:32:36.585 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] Token/tag mapping: [(There, True), (were, False), (no, False), (echocardiographic, False), (signs, False), (of, False), (right, False), (heart, False), (strain, False), (., False)]


Case D: checking for UNRESOLVED
--------------------------------------------------
  (no entities matched — add a TargetRule for 'right heart strain' to exercise this case)
